# **Laboratorio #2 - Aprendizaje por Refuerzo**
* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Mónica Salvatierra - 22249
* Bianca Calderón - 22272

Link del repositorio: https://github.com/paulabaal12/LAB2-RL

## **Task 1: Diseño formal del MDP**

**Contexto:** Una empresa de gestión de infraestructura hospitalaria opera un sistema de ascensores en un edificio de cinco pisos. En horas críticas, el tiempo de espera de camillas y personal médico tiene consecuencias directas sobre la atención al paciente. La gerencia quiere optimizar la política de despacho de ascensores usando Programación Dinámica antes de comprometer presupuesto en un sistema de aprendizaje completo.

Su grupo ha sido contratado para modelar el problema como un MDP, implementar Policy Iteration y Value Iteration desde cero, comparar su comportamiento, y producir un dictamen técnico con recomendaciones concretas para la gerencia

## **1. Espacio de estados 𝒮**

La dinámica de un MDP se describe mediante $p(s',r\mid s,a)$, que aparece en el operador de Bellman $\mathcal{T}^\pi V(s) = \sum_a \pi(a\mid s)\sum_{s',r} p(s',r\mid s,a)[r+\gamma V(s')]$. Que esta probabilidad dependa solo de $(s_t,a_t)$ y no del historial es la propiedad de Markov.

$$
p(s_{t+1},r_t \mid s_t,a_t,s_{t-1},a_{t-1},\dots) = p(s_{t+1},r_t \mid s_t,a_t)
$$

Si el estado del ascensor no cumpliera esto, por ejemplo si hiciera falta saber de dónde venía la cabina o cuánto tiempo lleva pendiente una llamada, $p(s'\mid s,a)$ quedaría mal definida y Policy/Value Iteration no aplicarían tal como se vieron en clase. Por eso el criterio para incluir o no una variable es siempre el mismo, si es necesaria para determinar $(s_{t+1}, r_t)$ sin depender de cómo se llegó ahí.

Se propone el siguiente estado.

$$
s_t = (f_t, d_t, C_t, E_t, o_t)
$$

- $f_t \in \{1,2,3,4,5\}$ es el piso actual de la cabina.
- $d_t \in \{\text{subiendo}, \text{bajando}, \text{detenido}\}$ es la dirección de movimiento.
- $C_t \subseteq \{1,\dots,5\}$ son los pisos con llamada pendiente, ya sea botón de piso o destino dentro de la cabina.
- $E_t \subseteq C_t$ son las llamadas marcadas como emergencia, activadas desde Urgencias, Quirófano, UCI, o mediante un botón de prioridad médica.
- $o_t \in \{0,1,\dots,o_{max}\}$ es el nivel de ocupación de la cabina.

### Justificación de las variables incluidas

$f_t$ es indispensable porque la recompensa y la transición dependen de la posición actual. $d_t$ evita que dos situaciones distintas, como subir o bajar por el mismo piso, colapsen al mismo estado, lo que rompería Markov. $C_t$ es la variable central porque sin llamadas pendientes no hay nada que optimizar. $E_t$ es obligatoria porque el contrato exige priorizar emergencias. $o_t$ importa porque una camilla ocupa espacio y puede impedir que la cabina recoja más pasajeros.

### Variables omitidas, supuesto y consecuencia

1. Número exacto de personas esperando en cada piso. Se asume que toda llamada pendiente pesa igual, así que el modelo no distingue un piso con una persona de uno saturado de personal médico, lo que en horas críticas puede subestimar la urgencia relativa entre pisos.

2. Tiempo exacto transcurrido desde que se generó cada llamada. Se aproxima con el costo acumulado en la recompensa en lugar de guardarlo en el estado, así que dos llamadas iguales en $C_t$ son indistinguibles aunque una lleve más tiempo esperando, lo que puede ocultar casos de starvation, llamadas que nunca llegan a ser atendidas.

3. Coordinación con otras cabinas del edificio. Se modela una sola cabina, asumiendo que cada ascensor puede tratarse como una unidad independiente. Se pierde la sinergia de asignar la llamada más cercana entre varias cabinas, una limitación a resolver antes de invertir en el sistema completo.

4. El momento del día, hora crítica u hora normal. El efecto se modela como una mayor tasa de llegada de llamadas en $p(s'\mid s,a)$ y no como variable de estado. Si la política debiera comportarse cualitativamente distinto en hora crítica, esto sería una limitación a revisar.

5. Condición mecánica o de mantenimiento del ascensor. Se asume operación nominal, así que el mantenimiento predictivo queda fuera del alcance de esta política.


## **2. Espacio de acciones 𝒜**

$$
\mathcal{A} = \{\text{Subir}, \text{Bajar}, \text{Atender}, \text{Esperar}\}
$$

Subir y Bajar mueven la cabina un piso. Atender detiene la cabina, abre puertas y resuelve la llamada del piso actual. Esperar mantiene la cabina quieta sin atender, útil para posicionarse estratégicamente cerca de un piso con alta probabilidad histórica de emergencias.

### ¿Discreto o continuo?

Debe ser discreto. Los pisos son discretos y las decisiones de despacho también lo son. Un control continuo de velocidad o aceleración corresponde al controlador de bajo nivel del ascensor, no a la política de despacho. Además, Policy Improvement, $\pi'(s) = \arg\max_a \sum_{s',r} p(s',r\mid s,a)[r+\gamma V^\pi(s')]$, y Value Iteration, $V_{k+1}(s) \leftarrow \max_a \sum_{s',r} p(s',r\mid s,a)[r+\gamma V_k(s')]$, recorren $\mathcal{A}$ con un máximo o argmax que solo es computable si es finito.

### Restricciones por estado $\mathcal{A}(s)$

Subir no está disponible si $f_t=5$. Bajar no está disponible si $f_t=1$. Atender solo está disponible si $f_t \in C_t$. Estas restricciones se modelan como soporte cero fuera de $\mathcal{A}(s)$, y esto importa porque el máximo o argmax de Value Iteration y Policy Improvement deben tomarse sobre $\mathcal{A}(s)$. De lo contrario el algoritmo podría elegir una acción físicamente imposible, como Subir desde el piso 5.

La disciplina de barrido, seguir en la misma dirección mientras haya llamadas pendientes adelante, no se impone como restricción dura, sino que se deja como comportamiento a aprender vía la recompensa. Así la política puede invertir dirección de inmediato para atender una emergencia detrás de la cabina, algo que una restricción rígida impediría.

## **3. Función de recompensa $r(s,a,s')$**

$$
r(s,a,s') = -w_1\cdot n_{normal}(s') - w_2\cdot n_{emerg}(s') + w_3\cdot \mathbb{1}[\text{atendida normal}] + w_4\cdot \mathbb{1}[\text{atendida emergencia}] - w_5\cdot \mathbb{1}[a\in\{\text{Subir},\text{Bajar}\}]
$$

$n_{normal}(s')$ y $n_{emerg}(s')$ son la cantidad de llamadas normales y de emergencia pendientes en $s'$.

### Relación con $p(s',r\mid s,a)$

La presentación deja la recompensa como parte de $p(s',r\mid s,a)$ sin asumir que es determinista. Aquí sí lo es, así que $p(s',r\mid s,a) = p(s'\mid s,a)\cdot \mathbb{1}[r=r(s,a,s')]$, y el operador de Bellman se reduce a

$$
\sum_{s'} p(s'\mid s,a)\left[r(s,a,s')+\gamma V(s')\right].
$$

### Valores propuestos

Se propone $-1$ por llamada normal pendiente y $-8$ por llamada de emergencia pendiente. Se suma $+15$ al atender una normal y $+100$ al atender una emergencia. El movimiento cuesta $-2$ y esperar cuesta $-0.5$, para que nunca convenga no hacer nada.

### Ponderación

Se ordena $w_2, w_4 \gg w_1, w_3 \gg w_5$, siguiendo el orden de prioridad del contrato. La seguridad del paciente va primero, el tiempo de espera general después, y la energía al final.

### Consecuencias de ponderar mal

Si las emergencias no dominan, el agente preferiría atender varias llamadas normales antes que una emergencia lejana, contradiciendo el contrato. Si $w_5$ es muy alta, la política evita moverse y deja llamadas sin atender. Si $w_5$ es cero, genera movimientos redundantes que gastan energía sin mejorar la espera. Si $w_1$ y $w_3$ fueran nulos, el sistema dejaría esperando indefinidamente a pacientes no urgentes.

## **4. Función de transición $p(s' \mid s,a)$**

Como en la Sección 3, $p(s',r\mid s,a)$ se reduce aquí a $p(s'\mid s,a)$, la parte con incertidumbre real que alimenta $\mathcal{T}^\pi$ y $\mathcal{T}^*$.

### ¿Determinista o estocástico?

La dinámica física es determinista. Dado $(f_t,d_t)$ y la acción, se sabe con certeza a qué piso se llega. La estocasticidad viene de afuera, de la llegada impredecible de nuevas llamadas normales, de emergencia y de destinos dentro de la cabina, además de variabilidad en el tiempo de embarque de una camilla. El sistema completo es estocástico.

### Transiciones concretas

**(a)** $s=(f{=}2,d{=}\text{subiendo},C{=}\{4\},E{=}\emptyset,o{=}0)$, $a=$ Subir.
- $p(f{=}3,d{=}\text{subiendo},C{=}\{4\},E{=}\emptyset,o{=}0 \mid s,a) = 0.85$ (no llega ninguna llamada nueva)
- $p(f{=}3,d{=}\text{subiendo},C{=}\{4,1\},E{=}\emptyset,o{=}0 \mid s,a) = 0.10$ (llega una nueva llamada normal, por ejemplo en el piso 1)
- $p(f{=}3,d{=}\text{subiendo},C{=}\{4\},E{=}\{3\},o{=}0 \mid s,a) = 0.05$ (llega una emergencia en el piso 3 mientras la cabina asciende)

**(b)** $s=(f{=}4,d{=}\text{subiendo},C{=}\{4\},E{=}\emptyset,o{=}0)$, $a=$ Atender.
- $p(f{=}4,d{=}\text{detenido},C{=}\emptyset,E{=}\emptyset,o{=}1 \mid s,a) = 0.90$ (la llamada se resuelve, sube 1 pasajero, sin nuevas llamadas)
- $p(f{=}4,d{=}\text{detenido},C{=}\{2\},E{=}\emptyset,o{=}1 \mid s,a) = 0.07$ (se resuelve la llamada y aparece una nueva llamada normal en el piso 2)
- $p(f{=}4,d{=}\text{detenido},C{=}\emptyset,E{=}\{5\},o{=}1 \mid s,a) = 0.03$ (se resuelve la llamada y aparece una emergencia en el piso 5)

**(c)** $s=(f{=}3,d{=}\text{detenido},C{=}\{3\},E{=}\{3\},o{=}0)$, $a=$ Atender, emergencia en el piso actual.
- $p(f{=}3,d{=}\text{detenido},C{=}\emptyset,E{=}\emptyset,o{=}2 \mid s,a) = 0.95$ (se atiende la emergencia, se asume que el traslado de camilla ocupa dos unidades de espacio de cabina)
- $p(f{=}3,d{=}\text{detenido},C{=}\{3\},E{=}\{3\},o{=}0 \mid s,a) = 0.05$ (el embarque de la camilla toma más tiempo del previsto y la emergencia permanece activa un paso más)

**(d)** $s=(f{=}1,d{=}\text{detenido},C{=}\emptyset,E{=}\emptyset,o{=}0)$, $a=$ Esperar.
- $p(f{=}1,d{=}\text{detenido},C{=}\emptyset,E{=}\emptyset,o{=}0 \mid s,a) = 0.80$ (no llega ninguna llamada nueva)
- $p(f{=}1,d{=}\text{detenido},C{=}\{x\},E{=}\emptyset,o{=}0 \mid s,a) = 0.15$ (llega una llamada normal en algún piso $x\neq 1$)
- $p(f{=}1,d{=}\text{detenido},C{=}\emptyset,E{=}\{y\},o{=}0 \mid s,a) = 0.05$ (llega una emergencia en algún piso $y$)

Las magnitudes asumen una tasa de llegada normal de 10 a 15% por paso y de emergencia de 3 a 5% por paso en horas críticas, dado que el contexto señala que las consecuencias sobre la atención al paciente son directas en esos horarios.

## **5. Factor de descuento $\gamma$**

Se propone $\gamma = 0.9$.

### Por qué debe cumplirse $\gamma < 1$

$\mathcal{T}^\pi$, y de forma análoga $\mathcal{T}^*$, es una contracción de factor $\gamma$ bajo la norma $\|\cdot\|_\infty$, con $\|\mathcal{T}V-\mathcal{T}V'\|\le\gamma\|V-V'\|$. Solo con $\gamma<1$ se garantiza que $\gamma^k\to 0$ y, por el teorema del punto fijo de Banach, que $\mathcal{T}^k V$ converge a $V^\pi$ (o $V^*$) desde cualquier $V_0$. Cualquier $\gamma$ en ese rango garantiza convergencia. El valor exacto es entonces una decisión de diseño y no un requisito matemático.

### Justificación de dominio

El despacho de ascensores es una tarea continua, ya que el hospital opera de forma ininterrumpida. Un valor moderado alto como 0.9 permite que la política mire unos pasos adelante, lo suficiente para preferir una emergencia cercana sobre una llamada normal inmediata, sin sobrevalorar planificación especulativa irrelevante en un sistema donde cada paso dura segundos.

### $\gamma$ cercano a 0 versus cercano a 1

Cuando $\gamma$ tiende a 0, el agente es miope y solo valora el beneficio inmediato, sin capacidad de planear una ruta, lo que produce reversas de dirección innecesarias. Cuando $\gamma$ tiende a 1, el agente puede posicionarse estratégicamente cerca de Urgencias anticipando futuras emergencias, pero la tasa de contracción decae más lento, así que Policy/Value Iteration requieren muchas más iteraciones para converger, y el agente puede diferir la atención de llamadas normales especulando con recompensas futuras, generando starvation.

## **Referencias bibliográficas**

1. Bellman, R. (1957). Dynamic programming. Princeton University Press.

2. Bertsekas, D. P. (2012). Dynamic programming and optimal control (4th ed., Vol. II). Athena Scientific.

3. Puterman, M. L. (1994). Markov decision processes: Discrete stochastic dynamic programming. John Wiley & Sons.

4. Sutton, R. S., & Barto, A. G. (2018). Reinforcement learning: An introduction (2nd ed.). MIT Press.